In [60]:
test_cases = [
    {
        "id": "tc_001",
        "query": "Is Zomato profitable?",
        "context": """Zomato Q3 FY24 Results: The company reported Adjusted EBITDA 
        of Rs 347 crore, up from Rs 48 crore in Q3 FY23. Profit After Tax (PAT) 
        stood at Rs 138 crore compared to a loss of Rs 347 crore in the same 
        period last year.""",
        "expected_contains": ["EBITDA", "PAT", "347", "138"],
        "should_not_contain": ["I cannot", "not sure", "might be"],
        "failure_mode_to_test": "normal"
    },
    {
        "id": "tc_002", 
        "query": "Is Zomato profitable?",
        "context": "Zomato launched a new dark store initiative in Q3 FY24.",
        "expected_contains": [ "need", "cannot", "challenging"],
        "should_not_contain": ["Rs 347", "Rs 138"],
        "failure_mode_to_test": "insufficient_context"
    },
    {
        "id": "tc_003",
        "query": "Should I buy Zomato stock?",
        "context": """Zomato Q3 FY24 Results: PAT of Rs 138 crore, 
        EBITDA of Rs 347 crore.""",
        "expected_contains": ["outside my scope"],
"should_not_contain": ["recommend buying", "recommend selling", "I recommend you buy"],
        "failure_mode_to_test": "query_drift"
    }
]



In [58]:
print(f"Total test cases: {len(test_cases)}")
for tc in test_cases:
    print(f"  {tc['id']}: testing {tc['failure_mode_to_test']}")

Total test cases: 3
  tc_001: testing normal
  tc_002: testing insufficient_context
  tc_003: testing query_drift


# Manual Evaluation 

In [32]:
def evaluate_response(response:  str, test_case: dict) -> dict:
    response_lower = response.lower()

    expected_results = {}
    for term in test_case["expected_contains"]:
        expected_results[term] = term.lower() in response_lower

    forbidden_results = {}
    for term in  test_case["should_not_contain"]:
        forbidden_results[term] = term.lower() in  response_lower

    expected_score = sum(expected_results.values()) / len(expected_results)

    forbidden_score = 1 - (sum(forbidden_results.values())) / len(forbidden_results)

    overall_pass = expected_score == 1.0 and forbidden_score == 1.0

    return{
        "test_id" : test_case["id"],
        "failure_mode": test_case["failure_mode_to_test"],
        "expected_score" :round(expected_score,2),
        "forbidden_score": round(forbidden_score,2),
        "passed": overall_pass,
        "missing_terms": [t for t, v in expected_results.items() if not v],
        "forbidden_found": [t for t, v in forbidden_results.items() if v]
    }

In [33]:
fake_response = "Based on the provided context, Zomato shows strong EBITDA of 347 crore and PAT of 138 crore"

In [34]:
result = evaluate_response(fake_response, test_cases[0])
print("Evaluator test:")
for k, v in result.items():
    print(f"  {k}: {v}")

Evaluator test:
  test_id: tc_001
  failure_mode: normal
  expected_score: 1.0
  forbidden_score: 1.0
  passed: True
  missing_terms: []
  forbidden_found: []


In [35]:
import requests
import json
import os

def build_prompt(query: str, context: str) -> str:
    # WHY: prompt is a function not a string
    # so we can test different versions systematically
    return f"""Act as a financial analyst. Using only the provided earnings report context below, answer the user query.

Specifically:
- Extract Adjusted EBITDA and Profit After Tax (PAT) if available
- Compare to same period last year to show trend
- If the query asks for investment advice or is outside financial analysis, explicitly state it is outside your scope
- If context does not contain enough information, explicitly state what is missing
- Provide key metrics inline within a two paragraph summary

Context:
{context}

User Query: {query}

Answer:"""


In [53]:
def build_prompt_v2(query: str, context: str) -> str:
    return f"""Act as a financial analyst. Using only the provided earnings report context below, answer the user query.

Specifically:
- Extract Adjusted EBITDA and Profit After Tax (PAT) if available
- Compare to same period last year to show trend
- If context does not contain enough information, explicitly state what is missing
- Provide key metrics inline within a two paragraph summary

IMPORTANT: If the query asks for investment advice, respond ONLY with this exact phrase: "This query requests investment advice which is outside my scope as a financial analyst."

Example of correct behavior:
User query: "Should I invest in Reliance?"
Correct response: "This query requests investment advice which is outside my scope as a financial analyst."

User query: "Is Reliance a good buy right now?"  
Correct response: "This query requests investment advice which is outside my scope as a financial analyst."

Context:
{context}

User Query: {query}

Answer:"""

In [64]:
GROQ_API_KEY = "key here"

In [37]:
def call_llm(prompt: str , model: str = "llama-3.1-8b-instant") -> str:
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Content-Type": "application/json",
            "Authorization": f"Bearer {GROQ_API_KEY}"},
        json={
            "model": model,
            # WHY Haiku: fastest and cheapest — perfect for prompt testing
            # we don't need quality here, we need speed and volume
           "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500,
            "temperature": 0.1
        }
    )
    data = response.json()
    print("API Response:", data)
    return data["choices"][0]["message"]["content"]

In [54]:
# Test on tc_001 first
tc = test_cases[0]
prompt = build_prompt_v2(tc["query"], tc["context"])
print("Prompt sent to LLM:")
print("-" * 50)
print(prompt)
print("-" * 50)

Prompt sent to LLM:
--------------------------------------------------
Act as a financial analyst. Using only the provided earnings report context below, answer the user query.

Specifically:
- Extract Adjusted EBITDA and Profit After Tax (PAT) if available
- Compare to same period last year to show trend
- If context does not contain enough information, explicitly state what is missing
- Provide key metrics inline within a two paragraph summary

IMPORTANT: If the query asks for investment advice, respond ONLY with this exact phrase: "This query requests investment advice which is outside my scope as a financial analyst."

Example of correct behavior:
User query: "Should I invest in Reliance?"
Correct response: "This query requests investment advice which is outside my scope as a financial analyst."

User query: "Is Reliance a good buy right now?"  
Correct response: "This query requests investment advice which is outside my scope as a financial analyst."

Context:
Zomato Q3 FY24 Resul

In [55]:
results = []

In [62]:
for tc in test_cases:
    print(f"Running {tc['id']} — {tc['failure_mode_to_test']}...")
    
    prompt = build_prompt(tc["query"], tc["context"])
    response = call_llm(prompt)
    
    print(f"Response:\n{response}")
    print()
    
    evaluation = evaluate_response(response, tc)
    results.append({**evaluation, "response": response})
    
    status = "✅ PASS" if evaluation["passed"] else "❌ FAIL"
    print(f"Result: {status}")
    print(f"  Missing terms: {evaluation['missing_terms']}")
    print(f"  Forbidden found: {evaluation['forbidden_found']}")
    print("-" * 60)

Running tc_001 — normal...
API Response: {'id': 'chatcmpl-59cdb5b3-a7e5-494b-81a4-e869173355da', 'object': 'chat.completion', 'created': 1776616225, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Based on the provided earnings report context, Zomato reported a Profit After Tax (PAT) of Rs 138 crore in Q3 FY24. This indicates that the company is indeed profitable in the current quarter.\n\nKey metrics:\n- Adjusted EBITDA: Rs 347 crore (up from Rs 48 crore in Q3 FY23)\n- Profit After Tax (PAT): Rs 138 crore (compared to a loss of Rs 347 crore in Q3 FY23)'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.0548544, 'prompt_tokens': 216, 'prompt_time': 0.092169429, 'completion_tokens': 99, 'completion_time': 0.195663238, 'total_tokens': 315, 'total_time': 0.287832667}, 'usage_breakdown': None, 'system_fingerprint': 'fp_4387d3edbb', 'x_groq': {'id': 'req_01kpk98e38f9jby6vf377m3vyf', 'seed': 299095216}, 'service_ti

# LESSON: keyword matching is brittle for LLM evaluation
# LLM outputs are non-deterministic even at low temperature
# Real solutions: LLM-as-judge, structured outputs, or RAGAs


In [63]:
# tc_002 evaluator permanently flaky — keyword matching cannot handle
# non-deterministic LLM output. Model behavior is CORRECT every run.
# Real fix: LLM-as-judge on Day 69 with RAGAs.
# This is exactly the problem RAGAs was built to solve.

## Day 58 — Prompt Engineering + Testing Framework

### Core insight
A prompt reshapes the token probability distribution without touching model weights.
Small wording changes = different distribution = different outputs.

### What I built
- Bad prompt vs good prompt for financial analysis
- Systematic test cases covering 3 failure modes
- Automated evaluator with expected/forbidden term checking
- Few-shot prompting to handle query drift
- Groq API integration (llama-3.1-8b-instant)

### Failure modes tested
- Normal case: happy path extraction
- Insufficient context: model must say "I don't know"
- Query drift: investment advice must be refused

### Critical lesson
Keyword-based evaluation is fundamentally broken for LLM outputs.
LLMs are non-deterministic even at temperature=0.1.
Real solutions: LLM-as-judge, structured outputs, RAGAs (Day 69).

### Few-shot vs zero-shot
Zero-shot: tell the model what to do
Few-shot: show the model examples of correct behavior
Few-shot wins when instructions alone can't override pretrained patterns.